# Overview

This notebook id dedicated for evaluating NER model on a benchmark dataset.

# Step 0 - Setup
Run the code below to mount your Google Drive and most of the necessary packages to carry out the evaluation

**Action:**
No code changes required. When prompted, connect your Google account

In [1]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [2]:
%%capture
!cd /content/
!rm -rf ./CASM_utils/
!git clone -b master https://github.com/ay94/multilingual-ner.git
!pip install -e CASM_utils/
!cd /content/CASM_utils


import CASM_utils
import importlib
from CASM_utils import utils, ner
importlib.reload(ner)

In [3]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

import os
import pandas as pd

Mounted at /content/drive/


In [4]:
# direct the file handler to the data folder
FOLDER = '/content/drive/MyDrive/CASM/FAST/German/NER/Benchmark'
fh = utils.FileHandler(FOLDER)

 # Read NER Dataset --> the output should be list of words corresponding to list of labels
 Read NER data class, it contains three different functionalities to read NER data.
  The NER data in the literature normally have consistent internal structure and flexible external structure.
  The internal structure is that it comes in word-label pair, this is consistent across all datasets.
  The external structure normally differ from dataset to another, which is divided to three main categories:
  - Data that comes in one text file, the read_ner_file function can be used in this case.
  - Data that comes in text files split into, train, val and test, this type you can either read individual file separately or put them all in one folder and read_ner_directory function.
  - Data that comes in directory where the directory contians various text files divide by topic (e.g, AQMAR), this type of data normally wikipedia articles that has been scraped and preprocessed into named entities structure.
  - Finally data available on huggingface and this can be loaded using load_dataset function and pass it to the read_dataset class method.
  
  Most of the datasets fall under one of these types if your data is different you can add function to this class dedicated to your data.

xtreme

In [6]:
xtreme_label_map = {
    'O': 0, 'B-PER': 1, 'I-PER': 2,
    'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6
}
xtreme = ner.ReadNERData()
xtreme_words, xtreme_labels = xtreme.read_dataset('xtreme', xtreme_label_map, lang='PAN-X.de')

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

germeval_14

In [21]:
germeval_14_label_map = {
    'O': 0,
    'B-LOC': 1, 'I-LOC': 2, 'B-LOCderiv': 3, 'I-LOCderiv': 4, 'B-LOCpart': 5, 'I-LOCpart': 6,
    'B-ORG': 7, 'I-ORG': 8, 'B-ORGderiv': 9, 'I-ORGderiv': 10, 'B-ORGpart': 11, 'I-ORGpart': 12,
    'B-OTH': 13, 'I-OTH': 14, 'B-OTHderiv': 15, 'I-OTHderiv': 16, 'B-OTHpart': 17, 'I-OTHpart': 18,
    'B-PER': 19, 'I-PER': 20, 'B-PERderiv': 21, 'I-PERderiv': 22, 'B-PERpart': 23, 'I-PERpart': 24
}



germeval_14 = ner.ReadNERData()
germeval_14_words, germeval_14_labels = germeval_14.read_dataset('germeval_14', germeval_14_label_map)

Generating test Split


  0%|          | 0/5100 [00:00<?, ?it/s]

### Check for dataset alignment
The first thing to do after loading the data is to check that it is aligned with the standard annotation scheme using check_labels function. NER datasets have various annotation schemes, the standard one we are interested in is the conll annotation scheme where the data should be divided into, *LOC*, *PERS*, *ORG*, *MISC* entities and each entity have BI boudary (e.g, B-LOC, I-LOC) and outside named entity O. This is the standard annotation scheme we are aiming for and some dataset comes with fine grained annotations or even different labels. This requires realigning the dataset labels to the standard scheme by defining a dataset label alignment dictionary and use the align_dataset function.

xtreme

In [22]:
ner.check_labels(xtreme_labels)

{'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O'}

collection3

In [23]:
print(ner.check_labels(germeval_14_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC
germeval_14_label_alignment = {
            "B-LOCderiv": "B-LOC", "I-LOCderiv": "I-LOC",
            "B-LOCpart": "B-LOC", "I-LOCpart": "I-LOC",
            "B-ORGderiv": "B-ORG", "I-ORGderiv": "I-ORG",
            "B-ORGpart": "B-ORG", "I-ORGpart": "I-ORG",
            "B-PERderiv": "B-ORG", "I-PERderiv": "I-ORG",
            "B-PERpart": "B-ORG", "I-PERpart": "I-ORG",
            "B-OTHderiv": "O", "I-OTHderiv": "O",
            "B-OTHpart": "O", "I-OTHpart": "O",
            "B-OTH": "O", "I-OTH": "O",



}
# Align the dataset labels to the standard labels
germeval_14_labels = ner.align_dataset(germeval_14_labels, germeval_14_label_alignment)
print(ner.check_labels(germeval_14_labels))


{'B-ORGderiv', 'B-PER', 'B-PERderiv', 'I-ORGpart', 'I-PERpart', 'B-LOCpart', 'B-OTHderiv', 'B-ORGpart', 'I-LOCderiv', 'O', 'B-LOC', 'I-ORG', 'I-OTH', 'B-OTHpart', 'B-LOCderiv', 'B-OTH', 'I-LOC', 'B-ORG', 'B-PERpart', 'I-PER'}
{'O', 'B-LOC', 'I-LOC', 'I-ORG', 'B-ORG', 'B-PER', 'I-PER'}


# Model Evaluation
Model evaluation is dvided into three steps:
- Loading the model using get_model funtion
- Generating the evaluation benchmark using generate_evaluation_data function
- Apply the model to the benchmakr and compute the performance using eval_fn

All of these steps can be achieved by calling evaluate_model function. It is worth noting that all models comes with their own labeling scheme and some models have different annotation scheme from the standard one we are using, this requires using label alignment dictionary to align the model's output.

In [32]:
model_name = "fhswf/bert_de_ner"
model_name_output = 'fhswf-bert-GermEval2014'
model_evaluation = ner.ModelEvaluation(model_name, germeval_14_label_alignment)

In [33]:
fh.create_folder(f'outputs/{model_name_output}')

Folder 'outputs/fhswf-bert-GermEval2014' already exists.


In [34]:
model_evaluation.model.config.id2label

{0: 'O',
 1: 'B-LOC',
 2: 'B-LOCderiv',
 3: 'B-LOCpart',
 4: 'B-ORG',
 5: 'B-ORGderiv',
 6: 'B-ORGpart',
 7: 'B-OTH',
 8: 'B-OTHderiv',
 9: 'B-OTHpart',
 10: 'B-PER',
 11: 'B-PERderiv',
 12: 'B-PERpart',
 13: 'I-LOC',
 14: 'I-LOCderiv',
 15: 'I-LOCpart',
 16: 'I-ORG',
 17: 'I-ORGderiv',
 18: 'I-ORGpart',
 19: 'I-OTH',
 20: 'I-OTHderiv',
 21: 'I-OTHpart',
 22: 'I-PER',
 23: 'I-PERderiv',
 24: 'I-PERpart'}

#### xtreme

In [35]:
data_name = "xtreme"
xtreme_evaluation_output = model_evaluation.evaluate_model(xtreme_words, xtreme_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [36]:
xtreme_seqeval = xtreme_evaluation_output.get_classification('Seqeval')
xtreme_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6025,0.7031,0.6489,4961
1,ORG,0.6270,0.3558,0.4540,4157
2,PER,0.7633,0.7408,0.7519,4750
3,micro,0.6652,0.6119,0.6374,13868
4,macro,0.6643,0.5999,0.6183,13868
5,weighted,0.6649,0.6119,0.6258,13868


In [37]:
xtreme_sklearn = xtreme_evaluation_output.get_classification('Sklearn')
xtreme_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.6624,0.7555,0.7059,4961
1,B-ORG,0.7769,0.4198,0.5451,4157
2,B-PER,0.8738,0.8366,0.8548,4750
3,I-LOC,0.6059,0.3438,0.4387,2289
4,I-ORG,0.9202,0.4006,0.5582,6043
5,I-PER,0.9572,0.7503,0.8412,6792
6,O,0.8969,0.9921,0.9421,68654
7,accuracy,0.8795,97646,None,None
8,macro,0.8133,0.6427,0.6980,97646
9,weighted,0.8776,0.8795,0.8664,97646


In [38]:
xtreme_seqeval.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-seqeval.csv'),
    index=False
)
xtreme_sklearn.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-sklearn.csv'),
    index=False
)

#### germeval_14

In [39]:
data_name = "germeval_14"
germeval_14_evaluation_output = model_evaluation.evaluate_model(germeval_14_words, germeval_14_labels)

  0%|          | 0/319 [00:00<?, ?it/s]

In [40]:
germeval_14_seqeval = germeval_14_evaluation_output.get_classification('Seqeval')
germeval_14_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.8651,0.8990,0.8817,2376
1,ORG,0.7642,0.7466,0.7553,1385
2,PER,0.8978,0.9170,0.9073,1639
3,micro,0.8503,0.8654,0.8577,5400
4,macro,0.8424,0.8542,0.8481,5400
5,weighted,0.8492,0.8654,0.8571,5400


In [41]:
germeval_14_sklearn = germeval_14_evaluation_output.get_classification('Sklearn')
germeval_14_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.8825,0.9108,0.8964,2376
1,B-ORG,0.8106,0.7726,0.7911,1385
2,B-PER,0.9187,0.9304,0.9245,1639
3,I-LOC,0.8507,0.7427,0.7930,307
4,I-ORG,0.8031,0.7394,0.7699,706
5,I-PER,0.9323,0.9671,0.9494,912
6,O,0.9936,0.9940,0.9938,89173
7,accuracy,0.9847,96498,None,None
8,macro,0.8845,0.8653,0.8740,96498
9,weighted,0.9846,0.9847,0.9846,96498


In [42]:
germeval_14_seqeval.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-seqeval.csv'),
    index=False
)
germeval_14_sklearn.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-sklearn.csv'),
    index=False
)
